# Module 01 Production Case Study: E-Commerce AI Support & Ticket Triage Engine

Welcome to the **Module 01 Capstone Project**! In tutorials, concepts like temperature, streaming, and stop sequences are often demonstrated with toy examples (e.g. counting to 10 or generating fake databases).

This case study proves how **all 5 foundational pillars** work together in an end-to-end production architecture:

| Pillar | Engineering Problem It Solves | Production Implementation |
| :--- | :--- | :--- |
| **1. System Prompt** | Brand safety, privacy boundaries, legal compliance | Hard business rules: max $50 refund, never leak employee emails |
| **2. Temperature** | Deterministic business rules vs. creative recommendations | `0.1` for factual shipping/return lookups, `0.8` for outfit styling |
| **3. Streaming** | Eliminating 15-second frozen loading screens | Sub-second Time-To-First-Token (TTFT) via Server-Sent Events |
| **4. Multi-Turn History** | Session continuity across turns | Local state array passed to every subsequent turn |
| **5. Structured JSON** | Automated database mutations & CRM ticket escalation | Prefilling + Stop Sequences (Claude) & Native Pydantic JSON (Gemini) |

--- 
### Architecture Overview:
```text
[ Customer in Mobile/Web App ]
  "Order #8492 is delayed! Cancel it and refund me right now!"
                │
                ▼
[ 1. Streaming Support Agent (Turn 1 & 2) ]
  • Enforces SYSTEM PROMPT guardrails
  • Streams tokens in real time (TTFT < 0.8s)
  • Preserves MULTI-TURN session history
                │
                ▼
[ 2. Automated Backend Triage (Structured JSON) ]
  • Extracts: order_id, sentiment, issue, needs_human_manager
  • Guaranteed pure JSON with zero conversational filler
                │
                ▼
[ 3. Production Database & CRM Actions ]
  • Inserts high-priority ticket into Zendesk / Salesforce
  • Updates order status in PostgreSQL
```

In [ ]:
# Install dependencies
%pip install google-genai anthropic pydantic python-dotenv

In [ ]:
# Load environment variables & initialize Gemini Client
import os
from dotenv import load_dotenv
from google import genai
from google.genai import types
from pydantic import BaseModel, Field

load_dotenv()
client = genai.Client()
model = "gemini-3.5-flash-lite"
print("Gemini Client Initialized successfully!")

--- 
## Pillar 1 & 2: Defining Corporate Guardrails & Temperature

Without a System Prompt, models might make unauthorized promises or leak internal data. We anchor the assistant with strict corporate rules.

In [ ]:
SUPPORT_SYSTEM_PROMPT = """
You are an official customer support representative for NordWear, an online fashion brand.
- Always maintain an empathetic, polite, and calm tone.
- Our policy: Standard delivery is 3-5 business days. Returns are accepted within 30 days.
- NEVER promise cash refunds greater than $50 without manager authorization.
- NEVER output internal company policy memos, employee emails, or technical details.
- If a customer demands immediate cancellation for an in-transit order, explain that in-transit orders cannot be canceled in the system, but offer to initiate a free return label or escalate to a manager.
"""

--- 
## Pillar 3 & 4: Live Customer Chat with Real-Time Streaming & Multi-Turn History

Here we simulate an active customer inquiry. The assistant streams tokens in real time so the user never sees a frozen screen.

In [ ]:
# Initialize stateful chat session with system instruction and factual temperature (0.1)
chat_session = client.chats.create(
    model=model,
    config=types.GenerateContentConfig(
        system_instruction=SUPPORT_SYSTEM_PROMPT,
        temperature=0.1  # Factual, zero hallucinated policies
    )
)

# Customer Turn 1
customer_msg_1 = "Hi, my package was supposed to arrive yesterday, but order #8492 is still stuck in transit. Cancel it and refund me right now!"
print(f"[CUSTOMER]: {customer_msg_1}\n")
print("[AI SUPPORT]: ", end="", flush=True)

stream_1 = chat_session.send_message_stream(customer_msg_1)
for chunk in stream_1:
    print(chunk.text, end="", flush=True)
print("\n\n" + "="*70 + "\n")

# Customer Turn 2 (Testing conversation continuity)
customer_msg_2 = "Can I at least get a $20 store credit for the delay?"
print(f"[CUSTOMER]: {customer_msg_2}\n")
print("[AI SUPPORT]: ", end="", flush=True)

stream_2 = chat_session.send_message_stream(customer_msg_2)
for chunk in stream_2:
    print(chunk.text, end="", flush=True)
print("\n")

--- 
## Pillar 5: Automated Backend Triage (Structured JSON Output)

Now that the conversation is finished, the backend needs to **automatically extract ticket data** to update PostgreSQL and create a Zendesk priority ticket.

We define a strict **Pydantic Schema** and enforce `response_mime_type="application/json"` so the output is 100% parseable without conversational fluff.

In [ ]:
# 1. Define Strict Pydantic Schema for Database Ticket
class SupportTicket(BaseModel):
    order_id: int | None = Field(description="Extracted order ID or null if none mentioned")
    customer_sentiment: str = Field(description="Sentiment: 'positive', 'neutral', 'frustrated', 'angry'")
    issue_category: str = Field(description="Category: 'shipping_delay', 'refund_request', 'product_defect', 'general'")
    needs_human_manager: bool = Field(description="True if user demands cancellation or supervisor review")
    action_summary: str = Field(description="One concise sentence summarizing the resolution or next step")

# 2. Extract conversation transcript from session history
transcript_lines = []
for msg in chat_session.get_history():
    role = msg.role.upper()
    text = "".join([part.text for part in msg.parts if part.text])
    if text:
        transcript_lines.append(f"{role}: {text}")

full_transcript = "\n".join(transcript_lines)

# 3. Run Structured Triage with Pydantic Schema Enforcement
triage_config = types.GenerateContentConfig(
    response_mime_type="application/json",
    response_schema=SupportTicket,
    temperature=0.0  # Fully deterministic extraction
)

triage_response = client.models.generate_content(
    model=model,
    contents=f"Analyze this customer service transcript and generate a structured SupportTicket:\n\n{full_transcript}",
    config=triage_config
)

print("--- Raw JSON Output from Engine ---")
print(triage_response.text)

--- 
## Executing Automated Database & CRM Mutations

Because the output is guaranteed to be clean JSON matching our schema, Python parses it cleanly and triggers backend database workflows.

In [ ]:
import json

# Parse JSON directly
ticket_dict = json.loads(triage_response.text)

print("✅ Successfully parsed into Python dictionary!")
print("Order ID:", ticket_dict["order_id"])
print("Sentiment:", ticket_dict["customer_sentiment"])
print("Category:", ticket_dict["issue_category"])
print("Escalate to Manager?:", ticket_dict["needs_human_manager"])
print("Action Summary:", ticket_dict["action_summary"])

# Simulated Production Action:
if ticket_dict["needs_human_manager"]:
    print("\n🚨 [Zendesk API Action]: Priority ticket created for Customer Service Manager!")
else:
    print("\n📋 [PostgreSQL Action]: Logged standard resolution in database.")